# Genetic Algorithms

Yeh notebook [AI for Beginners Curriculum](http://github.com/microsoft/ai-for-beginners) ka hissa hai.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import math
import time

## Thodi Theory

**Genetic Algorithms** (GA) ek **evolutionary approach** par based hain, jisme population ke evolve hone ke methods use karke kisi given problem ka best solution nikala jata hai. Inhe 1975 mein [John Henry Holland](https://en.wikipedia.org/wiki/John_Henry_Holland) ne propose kiya tha.

Genetic Algorithms in ideas par based hain:
* Problem ke valid solutions ko **genes** ke roop mein represent kiya ja sakta hai.
* **Crossover** hamein do solutions ko mila kar ek naya valid solution paane deta hai.
* **Selection** ka use kisi **fitness function** ke zariye zyada optimal solutions chunne ke liye hota hai.
* **Mutation** isliye introduce kiya jata hai ki optimization destabilize ho aur hum local minimum se bahar nikal sakein.

Agar aap Genetic Algorithm implement karna chahte hain, toh aapko yeh chahiye hoga:

 * Apne problem solution ko **genes** $g\in\Gamma$ se code karne ka tareeka dhundhein.
 * Genes ke set $\Gamma$ par **fitness function** $\mathrm{fit}: \Gamma\to\mathbb{R}$ define karein. Chhoti function values better solution ke corresponding hongi.
 * Do genes ko mila kar naya valid solution paane ke liye **crossover** mechanism define karein $\mathrm{crossover}: \Gamma^2\to\Gamma$.
 * **Mutation** mechanism define karein $\mathrm{mutate}: \Gamma\to\Gamma$.
Kai cases mein, crossover aur mutation genes ko numeric sequences ya bit vectors ki tarah manipulate karne ke liye kaafi simple algorithms hote hain.

Genetic algorithm ka specific implementation case-to-case alag ho sakta hai, lekin uska overall structure yeh hai:

1. Initial population select karein $G\subset\Gamma$
2. Randomly choose karein ki is step mein kaunsa operation perform karna hai: crossover ya mutation
3. **Crossover**:
  * Do genes randomly choose karein $g_1, g_2 \in G$
  * Crossover compute karein $g=\mathrm{crossover}(g_1,g_2)$
  * Agar $\mathrm{fit}(g)<\mathrm{fit}(g_1)$ ya $\mathrm{fit}(g)<\mathrm{fit}(g_2)$ - toh population mein corresponding gene ko $g$ se replace kar dein.
4. **Mutation** - randomly ek gene $g\in G$ choose karein aur use $\mathrm{mutate}(g)$ se replace kar dein.
5. Step 2 se repeat karein, jab tak $\mathrm{fit}$ ki value kaafi chhoti na ho jaye, ya steps ki limit tak na pahunch jayein.

Genetic algorithms se aam taur par solve hone wale tasks:
1. Schedule optimization
1. Optimal packing
1. Optimal cutting
1. Exhaustive search ko speed up karna

## Problem 1: Treasure ka fair division

**Task**:
Do logon ko ek treasure mila hai jisme alag-alag size (aur usi hisaab se alag-alag keemat) ke diamonds hain. Unhe treasure ko aise do hisson mein baantna hai ki keemat ka difference 0 (ya minimum) ho.

**Formal definition**:
Hamare paas numbers ka ek set $S$ hai. Hamein ise do subsets $S_1$ aur $S_2$ mein divide karna hai, taaki
$$\left|\sum_{i\in S_1}i - \sum_{j\in S_2}j\right|\to\min$$
aur $S_1\cup S_2=S$, $S_1\cap S_2=\emptyset$ ho.

Sabse pehle, set $S$ define karte hain:

In [ ]:
N = 200
S = np.array([random.randint(1,10000) for _ in range(N)])
print(S)

Chaliye problem ke har possible solution ko ek binary vector $B\in\{0,1\}^N$ se encode karte hain, jahan $i$-th position par number yeh dikhata hai ki original set $S$ mein $i$-th number kis set ($S_1$ ya $S_2$) se belong karta hai. `generate` function aise random binary vectors generate karega.

In [ ]:
def generate(S):
    return np.array([random.randint(0,1) for _ in S])

b = generate(S)
print(b)

Ab `fit` function define karte hain jo solution ki "cost" calculate karta hai. Yeh do sets, $S_1$ aur $S_2$ ke sum ke beech ka difference hoga:

In [ ]:
def fit(B,S=S):
    c1 = (B*S).sum()
    c2 = ((1-B)*S).sum()
    return abs(c1-c2)

fit(b)

Ab hamein mutation aur crossover ke liye functions define karne hain:
* Mutation ke liye, hum ek random bit choose karenge aur use flip kar denge (0 ko 1 aur 1 ko 0 bana denge)
* Crossover ke liye, hum ek vector se kuch bits lenge aur doosre vector se kuch bits lenge. Hum wahi `generate` function use karenge taaki woh randomly decide kar sake ki input masks se kaun-kaun se bits lene hain.

In [ ]:
def mutate(b):
    x = b.copy()
    i = random.randint(0,len(b)-1)
    x[i] = 1-x[i]
    return x

def xover(b1,b2):
    x = generate(b1)
    return b1*x+b2*(1-x)

Chaliye solutions ki initial population $P$ banate hain, size `pop_size` ke saath:

In [ ]:
pop_size = 30
P = [generate(S) for _ in range(pop_size)]

Ab, evolution process chalane wala main function. `n` woh number hai jo evolution ke steps dikhata hai. Har step mein:
* 30% probability ke saath hum ek mutation karte hain, aur sabse kharab `fit` function wale element ko mutated element se replace kar dete hain
* 70% probability ke saath hum crossover karte hain

Yeh function best solution (best solution se corresponding gene) aur har iteration mein population mein minimum fit function ki history return karta hai.

In [ ]:
def evolve(P,S=S,n=2000):
    res = []
    for _ in range(n):
        f = min([fit(b) for b in P])
        res.append(f)
        if f==0:
            break
        if random.randint(1,10)<3:
            i = random.randint(0,len(P)-1)
            b = mutate(P[i])
            i = np.argmax([fit(z) for z in P])
            P[i] = b
        else:
            i = random.randint(0,len(P)-1)
            j = random.randint(0,len(P)-1)
            b = xover(P[i],P[j])
            if fit(b)<fit(P[i]):
                P[i]=b
            elif fit(b)<fit(P[j]):
                P[j]=b
            else:
                pass
    i = np.argmin([fit(b) for b in P])
    return (P[i],res)

(s,hist) = evolve(P)
print(s,fit(s))

In [ ]:
plt.plot(hist)
plt.show()

## Problem 2: N Queens Problem

**Task**:
Aapko $N$ queens ko $N\times N$ size ke chess board par aise rakhna hai ki woh ek-doosre par attack na karein.

Sabse pehle, is problem ko genetic algorithm ke bina, exhaustive search se solve karte hain. Hum board position ko list $L$ se represent kar sakte hain, jahan list mein $i$-th number $i$-th row mein queen ki horizontal position hai. Yeh kaafi clear hai ki har solution mein har row mein sirf ek queen hogi.

Hamara goal problem ka pehla solution dhundhna hoga, jiske baad hum search ruk jayenge. Aap is function ko aasaani se extend karke saari possible queen positions generate kar sakte hain.

In [ ]:
N = 8

def checkbeats(i_new,j_new,l):
    for i,j in enumerate(l,start=1):
        if j==j_new:
            return False
        else:
            if abs(j-j_new) == i_new-i:
                return False
    return True

def nqueens(l,N=8,disp=True):
    if len(l)==N:
        if disp: print(l)
        return True
    else:
        for j in range(1,N+1):
            if checkbeats(len(l)+1,j,l):
                l.append(j)
                if nqueens(l,N,disp): return True
                else: l.pop()
        return False
            
nqueens([],8)


Ab chaliye measure karte hain ki 20-queens problem ka solution nikalne mein kitna time lagta hai:

In [ ]:
%timeit nqueens([],20,False)

Ab hum isi problem ko solve karne ke liye genetic algorithm use karenge. Yeh solution [is blog post](https://kushalvyas.github.io/gen_8Q.html) se inspired hai.

Hum har solution ko $N$ length ki wahi list se represent karenge, aur `fit` function ke roop mein hum un queens ki sankhya lenge jo ek-doosre par attack karti hain:

In [ ]:
def fit(L):
    x=0
    for i1,j1 in enumerate(L,1):
        for i2,j2 in enumerate(L,1):
            if i2>i1:
                if j2==j1 or (abs(j2-j1)==i2-i1): x+=1
    return x

In [ ]:
def generate_one(N):
    x = np.arange(1,N+1)
    np.random.shuffle(x)
    return (x,fit(x))

def generate(N,NP):
    return [generate_one(N) for _ in range(NP)]

generate(8,5)

In [ ]:
def mutate(G):
    x=random.randint(0,len(G)-1)
    G[x]=random.randint(1,len(G))
    return G
    
def xover(G1,G2):
    x=random.randint(0,len(G1))
    return np.concatenate((G1[:x],G2[x:]))

xover([1,2,3,4],[5,6,7,8])

In [ ]:
def choose_rand(P):
    N=len(P[0][0])
    mf = N*(N-1)//2 # max fitness fn
    z = [mf-x[1] for x in P]
    tf = sum(z) # total fitness
    w = [x/tf for x in z]
    p = np.random.choice(len(P),2,False,p=w)
    return p[0],p[1]

def choose(P):
    def ch(w):
        p=[]
        while p==[]:
            r = random.random()
            p = [i for i,x in enumerate(P) if x[1]>=r]
        return random.choice(p)
    N=len(P[0][0])
    mf = N*(N-1)//2 # max fitness fn
    z = [mf-x[1] for x in P]
    tf = sum(z) # total fitness
    w = [x/tf for x in z]
    p1=p2=0
    while p1==p2:
        p1 = ch(w)
        p2 = ch(w)
    return p1,p2

Ab hum main evolutionary loop define karenge. Hum logic ko pichle example se thoda alag banayenge, taaki dikha sakein ki creativity use ki ja sakti hai. Hum tab tak loop karenge jab tak hamein correct solution (fitness function=0) nahi mil jata, aur har step mein hum current generation lekar usi size ki nayi generation banayenge. Yeh `nxgeneration` function se neeche diye steps mein hota hai:

1. Sabse kam fit solutions ko hata dein - iske liye `discard_unfit` function hai jo yeh kaam karta hai.
1. Generation mein kuch aur random solutions add karein.
1. `gen_size` size ki nayi generation ko, har naye gene ke liye neeche diye steps se taiyaar karein:
    - Do random genes select karein, fitness function ke proportion mein probability ke saath.
    - Ek crossover compute karein.
    - `mutation_prob` probability ke saath ek mutation apply karein.

In [ ]:
mutation_prob = 0.1

def discard_unfit(P):
    P.sort(key=lambda x:x[1])
    return P[:len(P)//3]

def nxgeneration(P):
    gen_size=len(P)
    P = discard_unfit(P)
    P.extend(generate(len(P[0][0]),3))
    new_gen = []
    for _ in range(gen_size):
        p1,p2 = choose_rand(P)
        n = xover(P[p1][0],P[p2][0])
        if random.random()<mutation_prob:
            n=mutate(n)
        nf = fit(n)
        new_gen.append((n,nf))
        '''
        if (nf<=P[p1][1]) or (nf<=P[p2][1]):
            new_gen.append((n,nf))
        elif (P[p1][1]<P[p2][1]):
            new_gen.append(P[p1])
        else:
            new_gen.append(P[p2])
        '''
    return new_gen
    
def genetic(N,pop_size=100):
    P = generate(N,pop_size)
    mf = min([x[1] for x in P])
    n=0
    while mf>0:
        #print("Generation {0}, fit={1}".format(n,mf))
        n+=1
        mf = min([x[1] for x in P])
        P = nxgeneration(P)
    mi = np.argmin([x[1] for x in P])
    return P[mi]

genetic(8)

In [ ]:
%timeit genetic(10)

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->